# 07 Calendar, course-design and Safe Exam Browser analyses (Reviewer 4, round 2)

Reproduces the analyses added in the second revision:

* **Calendar effects** (weekends, Indonesian public holidays and collective leave): Poisson regression of daily event counts with a B-spline semester trend, per faculty (Newey-West SE, 7 lags) and pooled (faculty-specific trends, SE clustered by date); structural metrics by day type.
* **Course design**: share of variance in log(1 + views per quiz attempt) explained by course and by student; per-course `attempt_viewed` self-loop rates.
* **Safe Exam Browser (SEB)**: quiz structure in courses with SEB configured versus other courses.

Input: the seven student partitions on Figshare (DOI 10.6084/m9.figshare.28341992). Each partition is downloaded, analysed and deleted in turn, so the notebook runs within the memory of a standard Colab session. Timestamps in the deposit are UTC and are converted to WIB (UTC+7).

In [ ]:
!pip -q install statsmodels scipy
import os, json, time
import numpy as np, pandas as pd, requests
import statsmodels.api as sm, statsmodels.formula.api as smf
from scipy.spatial.distance import jensenshannon

FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
OUT = "results_07"; os.makedirs(OUT, exist_ok=True)
META = requests.get("https://api.figshare.com/v2/articles/28341992", timeout=60).json()
URLS = {f["name"]: f["download_url"] for f in META["files"]}

# Public holidays and collective leave (cuti bersama), even semester 2022/2023
HOLIDAYS = pd.to_datetime([
    "2023-03-22", "2023-03-23",                      # Nyepi + collective leave
    "2023-04-07",                                    # Good Friday
    "2023-04-21", "2023-04-22", "2023-04-23",        # Eid al-Fitr + collective leave
    "2023-04-24", "2023-04-25", "2023-04-26",
    "2023-05-01",                                    # Labour Day
    "2023-05-18",                                    # Ascension Day
    "2023-06-01", "2023-06-02", "2023-06-04",        # Pancasila Day, Vesak + collective leave
])
AV, AS = "mod_quiz\\attempt_viewed", "mod_quiz\\attempt_started"
print(len(HOLIDAYS), "holiday dates")

In [ ]:
def load(fac):
    path = f"{fac}_Student.csv"
    with requests.get(URLS[path], stream=True, timeout=3600) as r:
        r.raise_for_status()
        with open(path, "wb") as fh:
            for ch in r.iter_content(1 << 23):
                fh.write(ch)
    df = pd.concat(pd.read_csv(path, usecols=["id", "event", "userid", "courseid", "timecreated"],
                               dtype={"id": "int64", "userid": "int32", "courseid": "int32", "event": "category"},
                               parse_dates=["timecreated"], chunksize=2_000_000), ignore_index=True)
    os.remove(path)
    n_raw = len(df)
    df = df.drop_duplicates().drop(columns="id")        # full-row deduplication
    df["event"] = df["event"].astype(str)
    df["date"] = (df["timecreated"] + pd.Timedelta(hours=7)).dt.normalize()   # UTC -> WIB
    d = pd.DatetimeIndex(df["date"])
    df["daytype"] = np.where(d.isin(HOLIDAYS), "holiday", np.where(d.dayofweek >= 5, "weekend", "weekday"))
    return df, n_raw

In [ ]:
def daily_counts(df):
    daily = df.groupby("date").size().rename("n").to_frame().iloc[1:-1]   # drop partial first/last days
    idx = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    daily = daily.reindex(idx, fill_value=0); daily.index.name = "date"
    daily["weekend"] = (daily.index.dayofweek >= 5).astype(int)
    daily["holiday"] = daily.index.isin(HOLIDAYS).astype(int)
    daily["t"] = np.arange(len(daily))
    return daily.reset_index()

def irr(m, k):
    ci = m.conf_int().loc[k]
    return [float(np.exp(m.params[k])), float(np.exp(ci[0])), float(np.exp(ci[1])), float(m.pvalues[k])]

def analyse(fac):
    t0 = time.time()
    df, n_raw = load(fac)
    res = {"faculty": fac, "rows_deposited": n_raw, "events": len(df)}
    # ---- calendar model on daily volume (Ramadan omitted: collinear with the semester trend)
    daily = daily_counts(df); daily["fac"] = fac
    m = smf.glm("n ~ weekend + holiday + bs(t, df=6)", data=daily,
                family=sm.families.Poisson()).fit(cov_type="HAC", cov_kwds={"maxlags": 7})
    null = smf.glm("n ~ bs(t, df=6)", data=daily, family=sm.families.Poisson()).fit()
    lab = np.where(daily.holiday == 1, "holiday", np.where(daily.weekend == 1, "weekend", "weekday"))
    res["calendar"] = {"days": len(daily), "weekend": irr(m, "weekend"), "holiday": irr(m, "holiday"),
                       "deviance_removed": float(1 - m.deviance / null.deviance),
                       "mean_daily": daily.groupby(lab)["n"].mean().round(0).to_dict()}
    # ---- directly-follows transitions within course-level cases
    df = df.sort_values(["courseid", "userid", "timecreated"], kind="mergesort").reset_index(drop=True)
    ev = df["event"].values; dt = df["daytype"].values; crs = df["courseid"].values
    same = (crs[1:] == crs[:-1]) & (df["userid"].values[1:] == df["userid"].values[:-1])
    s_ev, n_ev, s_dt, s_crs = ev[:-1][same], ev[1:][same], dt[:-1][same], crs[:-1][same]
    struct = {}
    for k in ["weekday", "weekend", "holiday", "all"]:
        msk = np.ones(len(s_ev), bool) if k == "all" else (s_dt == k)
        av_n = int(((ev == AV) & ((dt == k) if k != "all" else True)).sum())
        struct[k] = {"pct_self_loop": float(100 * (s_ev[msk] == n_ev[msk]).mean()),
                     "av_self_loop_rate": float(100 * ((s_ev[msk] == AV) & (n_ev[msk] == AV)).sum() / av_n)}
    acts = sorted(set(ev))
    def mix(k):
        v = pd.Series(ev[dt == k]).value_counts(); p = np.array([v.get(a, 0) for a in acts], float); return p / p.sum()
    res["mix_jsd"] = {"weekday_vs_weekend": float(jensenshannon(mix("weekday"), mix("weekend"), base=2)),
                      "weekday_vs_holiday": float(jensenshannon(mix("weekday"), mix("holiday"), base=2))}
    # ---- quiz attempts (segmented at attempt_started within a course-level case)
    q = df[df["event"].isin([AS, AV, "mod_quiz\\attempt_submitted"])].copy()
    q["case"] = q["courseid"].astype("int64") * 10_000_000 + q["userid"].astype("int64")
    q["att"] = q.groupby("case")["event"].transform(lambda s: (s == AS).cumsum())
    q = q[q["att"] > 0]; q["akey"] = q["case"].astype(str) + "_" + q["att"].astype(str)
    starts = q[q["event"] == AS].set_index("akey")
    att = pd.DataFrame({"views": q[q["event"] == AV].groupby("akey").size()}).reindex(starts.index, fill_value=0)
    att[["courseid", "userid", "daytype"]] = starts[["courseid", "userid", "daytype"]]
    qv = q[q["event"] == AV].sort_values(["akey", "timecreated"])
    qv = qv.assign(gap=qv.groupby("akey")["timecreated"].diff().dt.total_seconds()).dropna(subset=["gap"])
    for k in ["weekday", "weekend", "holiday", "all"]:
        a = att if k == "all" else att[att["daytype"] == k]; g = qv if k == "all" else qv[qv["daytype"] == k]
        struct[k].update({"attempts": int(len(a)), "median_views": float(a["views"].median()),
                          "median_gap_s": float(g["gap"].median())})
    res["structure_by_daytype"] = struct
    # ---- course design
    y = np.log1p(att["views"].values)
    def eta2(groups):
        gm = pd.Series(y).groupby(groups.values).transform("mean").values
        return float(((gm - y.mean()) ** 2).sum() / ((y - y.mean()) ** 2).sum())
    pc = att.groupby("courseid")["views"].agg(["median", "size"]); pc = pc[pc["size"] >= 20]["median"]
    av_c = pd.Series(crs[ev == AV]).value_counts()
    self_c = pd.Series(s_crs[(s_ev == AV) & (n_ev == AV)]).value_counts().reindex(av_c.index, fill_value=0)
    rate = (100 * self_c / av_c)[av_c >= 200]
    res["course_design"] = {"eta2_course": eta2(att["courseid"]), "eta2_student": eta2(att["userid"]),
                            "courses_20plus_attempts": int(len(pc)),
                            "course_median_views_p10_p50_p90": [float(pc.quantile(x)) for x in (.1, .5, .9)],
                            "course_median_views_min_max": [float(pc.min()), float(pc.max())],
                            "courses_200plus_av": int(len(rate)),
                            "pct_courses_av_selfloop_ge80": float((rate >= 80).mean() * 100),
                            "median_course_av_selfloop": float(rate.median())}
    # ---- Safe Exam Browser
    seb = set(df.loc[df["event"].str.startswith("quizaccess_seb"), "courseid"].unique())
    res["seb"] = {"seb_events": int(df["event"].str.startswith("quizaccess_seb").sum()), "n_seb_courses": len(seb),
                  "pct_attempts_in_seb_courses": float(att["courseid"].isin(seb).mean() * 100)}
    if seb:
        for lab_, inset in [("seb_courses", True), ("other_courses", False)]:
            cm = np.isin(s_crs, list(seb)) == inset
            avn = int(((ev == AV) & (np.isin(crs, list(seb)) == inset)).sum())
            sa = att["courseid"].isin(seb) == inset
            res["seb"][lab_] = {"courses": len(seb) if inset else int(df["courseid"].nunique() - len(seb)),
                                "attempts": int(sa.sum()), "median_views": float(att.loc[sa, "views"].median()),
                                "median_gap_s": float(qv.loc[qv["courseid"].isin(seb) == inset, "gap"].median()),
                                "av_self_loop_rate": float(100 * ((s_ev == AV) & (n_ev == AV) & cm).sum() / avn)}
    daily.to_csv(f"{OUT}/{fac}_daily.csv", index=False)
    json.dump(res, open(f"{OUT}/{fac}.json", "w"), indent=1)
    print(f"{fac}: done in {time.time()-t0:.0f}s")
    return res

In [ ]:
RESULTS = {}
for fac in FACULTIES:
    RESULTS[fac] = analyse(fac)

## Pooled calendar model (faculty-specific trends, standard errors clustered by date)

In [ ]:
D = pd.concat([pd.read_csv(f"{OUT}/{f}_daily.csv") for f in FACULTIES], ignore_index=True)
D["date_id"] = D["date"].astype("category").cat.codes
pooled = smf.glm("n ~ C(fac) + C(fac):bs(t, df=6) + weekend + holiday", data=D,
                 family=sm.families.Poisson()).fit(cov_type="cluster", cov_kwds={"groups": D["date_id"]})
print("days:", len(D))
for k in ["weekend", "holiday"]:
    print(k, "IRR %.3f [%.3f, %.3f]" % tuple(irr(pooled, k)[:3]))

## Summary tables (as reported in the manuscript)

In [ ]:
cal = pd.DataFrame({f: {"days": r["calendar"]["days"],
                         "weekend IRR": "%.2f [%.2f, %.2f]" % tuple(r["calendar"]["weekend"][:3]),
                         "holiday IRR": "%.2f [%.2f, %.2f]" % tuple(r["calendar"]["holiday"][:3]),
                         "deviance removed %": round(100 * r["calendar"]["deviance_removed"], 1)}
                     for f, r in RESULTS.items()}).T
display(cal)
rows = []
for f, r in RESULTS.items():
    for k in ["weekday", "weekend", "holiday"]:
        s = r["structure_by_daytype"][k]
        rows.append([f, k, s["attempts"], round(s["pct_self_loop"], 1), round(s["av_self_loop_rate"], 1),
                     s["median_views"], s["median_gap_s"]])
display(pd.DataFrame(rows, columns=["faculty", "day type", "attempts", "self-loop %", "attempt_viewed self-loop %",
                                    "median views", "median gap (s)"]))
display(pd.DataFrame({f: r["course_design"] for f, r in RESULTS.items()}).T)
display(pd.DataFrame({f: {k: v for k, v in r["seb"].items() if not isinstance(v, dict)} for f, r in RESULTS.items()}).T)
for f, r in RESULTS.items():
    if "seb_courses" in r["seb"]:
        print(f, "SEB:", r["seb"]["seb_courses"], "| other:", r["seb"]["other_courses"])